In [1]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 11.4 MB/s eta 0:00:00


In [2]:
import torch

print("GPU khả dụng:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU khả dụng: True
GPU: Tesla T4


In [3]:
from google.colab import files

uploaded = files.upload()

Saving mask_colab.zip to mask_colab.zip


In [6]:
from pathlib import Path
import zipfile

with zipfile.ZipFile("/content/mask_colab.zip") as archive:
    archive.extractall("/content/datasets")

print(list(Path("/content/datasets").iterdir()))

[PosixPath('/content/datasets/mask')]


In [7]:
dataset = Path("/content/datasets/mask")

assert (dataset / "images/train").is_dir()
assert (dataset / "labels/train").is_dir()

print("Đã tìm thấy dataset")

Đã tìm thấy dataset


In [8]:
yaml_path = dataset / "data.yaml"

yaml_path.write_text(
    """path: /content/datasets/mask
train: images/train
val: images/val
test: images/test
names:
  0: with_mask
  1: without_mask
  2: mask_incorrect
""",
    encoding="utf-8"
)

print(yaml_path.read_text())

path: /content/datasets/mask
train: images/train
val: images/val
test: images/test
names:
  0: with_mask
  1: without_mask
  2: mask_incorrect



In [9]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=640,
    batch=8,
    device=0,
    workers=2,
    patience=10,
    seed=42,
    project="/content/runs",
    name="mask_lab3"
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.175 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/mask/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ba7dc9d4c90>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04

In [11]:
best_path = Path(model.trainer.best)
print("Weights:", best_path)

trained_model = YOLO(str(best_path))
print("Các nhãn:", trained_model.names)

metrics = trained_model.val(
    data=str(yaml_path),
    split="test",
    imgsz=640,
    device=0
)

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

Weights: /content/runs/mask_lab3/weights/best.pt
Các nhãn: {0: 'with_mask', 1: 'without_mask', 2: 'mask_incorrect'}
Ultralytics 8.4.175 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,133,193 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2870.6±1504.7 MB/s, size: 534.9 KB)
val: Scanning /content/datasets/mask/labels/test... 127 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 127/127 468.4it/s 0.3s
val: New cache created: /content/datasets/mask/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.2it/s 6.6s
                   all        127        584      0.806      0.666      0.748      0.501
             with_mask        116        482      0.962      0.843      0.925      0.618
          without_mask         44         91      0.872      0.791      0.822      0.494
        mask_incorrect         10         11  

In [12]:
from google.colab import files

files.download(str(best_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>